# "THE POUR IS RIGHT" - Wine Pricer Capstone

## Week 8 Order of play

DAY 1: Modal and the SpecialistAgent  
DAY 2: RAG with Qdrant, FrontierAgent, Ensemble  
DAY 3: ScannerAgent, MessagingAgent  
DAY 4: The planning agent  
DAY 5: The user interface  

## DAY 2: RAG (Retrieval Augmented Generation)

So far we have improved models by training them. Today we improve a model at the moment we ask the question, with no training: before asking for a price, we look up the five most similar wines we already know and show the model what they cost. This follows `week8/day2.ipynb`.

Ed stored his products in Chroma. We use **Qdrant**, a different vector database. Where the two differ, the notes say what Ed's Chroma code does.

## Costs and time

| Step | Cost | Time |
|---|---|---|
| Building the vector store | free | about 17 minutes on this laptop |
| Qwen 3.8 with RAG, 200 wines | about 12 cents (Groq) | under a minute |
| GPT-5.1 with RAG, 200 wines | about 18 cents (OpenAI) | about a minute |
| Claude Sonnet 5 with RAG, 200 wines | about 55 cents (Anthropic) | about 2 minutes |
| The fine-tuned model on Modal, 200 wines | about 4 cents of Modal credit | about 4 minutes |
| Neural network and ensemble | free | seconds |

The GPT-5.1 and Sonnet cells are the ones with a noticeable cost. Each is marked, and you can skip either.

In [ ]:
import os
import re
import logging
import numpy as np
from dotenv import load_dotenv
from huggingface_hub import HfApi
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue, Range
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
import anthropic
import modal
from pricer.evaluator import evaluate
from pricer.items import Item

load_dotenv(override=True)
DB = "wines_vectorstore"
COLLECTION = "wines"

In [ ]:
username = HfApi().whoami()["name"]
dataset = f"{username}/wines_lite"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# The encoder: turning text into vectors

`all-MiniLM-L6-v2` is a small, free model that turns a piece of text into a list of 384 numbers, called an embedding or a vector. Texts with similar meaning get vectors that point in similar directions. It runs on this laptop, so nothing is sent anywhere.

It is a different kind of model from the ones we have used so far: it does not write text, it only produces these vectors.

In [ ]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
# Pass in a list of texts, get back a numpy array of vectors

vector = encoder.encode(["A silky Pinot Noir from Oregon with cherry and forest-floor notes"])[0]
print(vector.shape)
vector[:10]

# Create the Qdrant vector store

A vector database stores vectors and can find the ones closest to a new vector very quickly.

`QdrantClient(path=...)` runs Qdrant in local mode: the data lives in a folder on this laptop, with no server and no account. Ed's Chroma equivalent is `chromadb.PersistentClient(path=DB)`.

**One thing to know about local mode:** only one program can have the folder open at a time. If another notebook is using it, you will get an "already accessed by another instance" error. Close the other notebook's kernel, or run `client.close()` there.

In [ ]:
client = QdrantClient(path=DB)

## Create the collection and load the wines

A collection is Qdrant's name for a set of vectors, like a table in a database.

Qdrant makes you state two things that Chroma decides silently:

- **`size=384`**: how many numbers are in each vector. It must match the encoder.
- **`distance=Distance.COSINE`**: how closeness is measured. Cosine compares the direction of two vectors and ignores their length, which is the usual choice for text.

Each wine is stored as a **point** with three parts: an `id`, its `vector`, and a `payload`, which is any extra data we want back later. Ours holds the summary and the price, plus the country, variety, winery and points so that we can filter on them. In Chroma these are called documents and metadatas.

**The next cell takes about 17 minutes**, because the encoder has to read all 20,000 summaries on the CPU. It only runs once: if the collection is already there, it is skipped.

In [ ]:
if not client.collection_exists(COLLECTION):
    client.create_collection(COLLECTION, vectors_config=VectorParams(size=384, distance=Distance.COSINE))

    for i in tqdm(range(0, len(train), 1000)):
        batch = train[i: i+1000]
        vectors = encoder.encode([item.summary for item in batch])
        points = [
            PointStruct(
                id=i + j,
                vector=vector.tolist(),
                payload={
                    "summary": item.summary,
                    "price": item.price,
                    "category": item.category,
                    "variety": item.variety,
                    "winery": item.winery,
                    "points": item.points,
                },
            )
            for j, (item, vector) in enumerate(zip(batch, vectors))
        ]
        client.upsert(COLLECTION, points=points)

print(f"The collection holds {client.count(COLLECTION).count:,} wines")

# Let's visualize the vectors

We cannot picture 384 dimensions, so we use t-SNE, a technique that squeezes them down to 2 while trying to keep neighbours together. Each dot is a wine, coloured by country.

If the encoder has captured something real, wines from the same country should form clumps, even though nobody told it about countries. This takes a minute or two.

In [ ]:
MAXIMUM_DATAPOINTS = 4_000

points, _ = client.scroll(COLLECTION, limit=MAXIMUM_DATAPOINTS, with_vectors=True, with_payload=True)
vectors = np.array([point.vector for point in points])
documents = [point.payload["summary"] for point in points]
categories = [point.payload["category"] for point in points]

CATEGORIES = ['US', 'France', 'Italy', 'Spain', 'Portugal', 'Chile', 'Argentina', 'Austria', 'Germany', 'Australia', 'Other']
COLORS = ['blue', 'red', 'green', 'orange', 'purple', 'cyan', 'brown', 'pink', 'gold', 'magenta', 'gray']
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Country: {c}<br>{d[:60]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D view of the Qdrant vector store, coloured by country',
    width=1100,
    height=750,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

# Find similar wines

To search, we turn the new wine into a vector with the same encoder and ask Qdrant for the 5 nearest points. Each result comes back with a `score` (1.0 would be identical) and its payload.

Ed's Chroma equivalent is `collection.query(query_embeddings=..., n_results=5)`.

In [ ]:
def find_similars(item):
    vector = encoder.encode(item.summary)
    results = client.query_points(COLLECTION, query=vector.tolist(), limit=5, with_payload=True).points
    documents = [point.payload["summary"] for point in results]
    prices = [point.payload["price"] for point in results]
    return documents, prices

In [ ]:
print(test[0].summary)
print(f"\nActual price: ${test[0].price}\n")

vector = encoder.encode(test[0].summary)
for point in client.query_points(COLLECTION, query=vector.tolist(), limit=5, with_payload=True).points:
    wine = point.payload["summary"].split("\n")[0]
    print(f"score {point.score:.3f}   ${point.payload['price']:>6.2f}   {wine}")

The first test wine is a Calcareous Meritage at \$36. The search should return other wines from the same winery, at \$28 to \$40. That is exactly the kind of evidence a person would want before guessing a price.

## Something Qdrant is good at: filtered search

Qdrant can combine "find similar" with ordinary conditions on the payload, in one query. Ed's project did not use this.

Below, the same search is repeated twice: once restricted to French wines, and once to wines under \$20.

In [ ]:
french = Filter(must=[FieldCondition(key="category", match=MatchValue(value="France"))])
under_20 = Filter(must=[FieldCondition(key="price", range=Range(lt=20))])

for name, query_filter in [("French wines only", french), ("Wines under $20 only", under_20)]:
    print(f"\n{name}:")
    for point in client.query_points(COLLECTION, query=vector.tolist(), limit=3, query_filter=query_filter, with_payload=True).points:
        wine = point.payload["summary"].split("\n")[0]
        print(f"score {point.score:.3f}   ${point.payload['price']:>6.2f}   {point.payload['category']:8s} {wine}")

# RAG: give the model the similar wines

The prompt is the one from day 4 of week 6, with the five similar wines and their prices added underneath.

In [ ]:
def make_context(similars, prices):
    message = "For context, here are some other wines that might be similar to the wine you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related wine:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

def messages_for(item, similars, prices):
    message = f"Estimate the price of this bottle of wine in US dollars. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [ ]:
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]['content'])

## Look up the similar wines for the 200 test wines once

The evaluation calls the model for several wines at the same time. To keep the vector store out of that, we do all 200 searches here first and keep the results in a dictionary.

In [ ]:
contexts = {item.summary: find_similars(item) for item in tqdm(test[:200])}

In [ ]:
def get_price(reply):
    reply = (reply or "").replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

def cached(pricer):
    """Remember each wine's answer, so that the ensemble later can reuse it instead of paying for it again"""
    cache = {}
    def wrapper(item):
        if item.summary not in cache:
            cache[item.summary] = pricer(item)
        return cache[item.summary]
    wrapper.__name__ = pricer.__name__
    return wrapper

## Qwen 3.8 with RAG

With no context, Qwen 3.8 scored \$14.42 on day 4. On a 50-wine trial, adding the similar wines took it from about \$17 to about \$11, so expect a clear improvement here.

In [ ]:
@cached
def qwen_3__8_rag(item):
    documents, prices = contexts[item.summary]
    response = completion(model="groq/qwen/qwen3.8-27b", messages=messages_for(item, documents, prices), reasoning_effort="none")
    return get_price(response.choices[0].message.content)

In [ ]:
qwen_3__8_rag(test[0])

In [ ]:
evaluate(qwen_3__8_rag, test)

## An OpenAI model with RAG

Ed used GPT-5.1 for this step, so we do too. It costs about 18 cents for the 200 wines.

I first tried GPT-4.1 Nano here, the cheapest OpenAI model, and RAG made it worse (\$16.23 against \$15.10 without): it over-reacts to the similar wines and guesses far too high for some. A stronger model uses the context better.

Other models your key can use, with the cost of this 200-wine run. To try one, change `OPENAI_MODEL`.

| Model | Cost for 200 wines |
|---|---|
| `gpt-6-luna` | about 2 cents |
| `gpt-5-mini` | about 4 cents |
| `gpt-4.1-mini` | about 5 cents |
| `gpt-5.1` (Ed's choice) | about 18 cents |
| `gpt-6.1-sol` | about 35 cents |

Some of these models think before answering, and thinking is billed. `OPENAI_OPTIONS` switches it off or down for the models that allow it.

In [ ]:
OPENAI_MODEL = "gpt-5.1"

OPENAI_OPTIONS = {
    "gpt-5.1": {"reasoning_effort": "none"},
    "gpt-5-mini": {"reasoning_effort": "minimal"},
}

@cached
def openai_rag(item):
    documents, prices = contexts[item.summary]
    options = OPENAI_OPTIONS.get(OPENAI_MODEL, {})
    response = completion(model=f"openai/{OPENAI_MODEL}", messages=messages_for(item, documents, prices), **options)
    return get_price(response.choices[0].message.content)

In [ ]:
openai_rag(test[0])

In [ ]:
evaluate(openai_rag, test)

## Claude Sonnet 5 with RAG

**This is the cell that costs about 55 cents.** The prompt is about six times longer than on day 4 because of the five extra wines, and input tokens are what we pay for.

Sonnet 5 scored \$9.06 with no context. On a 50-wine trial it went from \$9.86 to \$8.00 with RAG.

To skip it, don't run the two cells below, and in the Ensemble section further down change `frontier = claude_sonnet_5_rag` to `frontier = openai_rag` or `frontier = qwen_3__8_rag`.

In [ ]:
claude = anthropic.Anthropic()

@cached
def claude_sonnet_5_rag(item):
    documents, prices = contexts[item.summary]
    response = claude.messages.create(
        model="claude-sonnet-5",
        max_tokens=16000,
        messages=messages_for(item, documents, prices),
        extra_body={"output_config": {"effort": "low"}},
    )
    if response.stop_reason == "refusal":
        return 0
    return get_price(next((block.text for block in response.content if block.type == "text"), "0"))

In [ ]:
evaluate(claude_sonnet_5_rag, test)

# The other two models

## The specialist: our fine-tuned model on Modal

This calls the service from day 1. `workers=1` sends one wine at a time, so that Modal keeps a single machine running instead of starting several. The first call may take a minute while the machine wakes up.

When I ran it, the service scored \$9.82. In Colab the same model scored \$10.10; small differences in how the model is loaded account for the gap.

In [ ]:
Pricer = modal.Cls.from_name("wine-pricer-service", "Pricer")
pricer = Pricer()

@cached
def specialist(item):
    return pricer.price.remote(item.summary)

In [ ]:
evaluate(specialist, test, workers=1)

## The deep neural network

This loads the weights saved on day 4 of week 6 (`deep_neural_network.pth`). `DeepNeuralNetworkInference` is a slimmed-down version of the training class that can price a wine without needing the training data. It should score \$11.23, as it did then.

In [ ]:
from pricer.deep_neural_network import DeepNeuralNetworkInference

runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")

@cached
def deep_neural_network(item):
    return runner.inference(item.summary)

In [ ]:
evaluate(deep_neural_network, test)

# The Ensemble

An ensemble combines several models' answers. The idea is that different models make different mistakes, and averaging cancels some of them out.

Ed weighted his models 80% frontier with RAG, 10% specialist, 10% neural network. We start with the same weights.

Every answer below comes from the caches filled above, so this section costs nothing.

In [ ]:
frontier = claude_sonnet_5_rag      # or openai_rag, or qwen_3__8_rag: use whichever scored best above

def ensemble(item):
    price1 = frontier(item)
    price2 = specialist(item)
    price3 = deep_neural_network(item)
    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1

In [ ]:
evaluate(ensemble, test)

## Does the weighting matter?

Here is the error for a few different weightings, and for each model alone.

Read this with care. We are choosing between weightings by looking at the test set, which is a mild form of cheating: with enough tries, something will look good by luck. Treat small differences as noise. The margin of error on 200 wines is nearly \$2.

On my 50-wine trial, no weighting beat Sonnet 5 with RAG on its own. The three models' mistakes turned out to be strongly related: when one over-prices a wine, the others tend to as well, which leaves little for averaging to cancel. Ed's ensemble only beat his best single model by 29 cents, too.

In [ ]:
wines = test[:200]
truth = np.array([item.price for item in wines])
f = np.array([frontier(item) for item in wines])
s = np.array([specialist(item) for item in wines])
n = np.array([deep_neural_network(item) for item in wines])

def error(prediction):
    return np.abs(prediction - truth).mean()

print(f"Frontier with RAG alone:   ${error(f):.2f}")
print(f"Specialist alone:          ${error(s):.2f}")
print(f"Neural network alone:      ${error(n):.2f}")
print()
for wf, ws, wn in [(0.8, 0.1, 0.1), (0.6, 0.2, 0.2), (0.5, 0.3, 0.2), (0.34, 0.33, 0.33), (0.5, 0.5, 0.0)]:
    print(f"Frontier {wf:.2f}, specialist {ws:.2f}, neural network {wn:.2f}:   ${error(wf*f + ws*s + wn*n):.2f}")

# The agents

The same pieces, packaged as agents for the rest of the week. Each takes a description of a wine and returns a price, logging what it does in its own colour.

- `FrontierAgent` (blue): RAG search in Qdrant, then Claude Sonnet 5. To use Qwen instead, create it with `FrontierAgent(client, model="groq/qwen/qwen3.8-27b")`.
- `NeuralNetworkAgent` (magenta): the deep neural network.
- `EnsembleAgent` (yellow): rewrites the description with the Preprocessor, asks the other three, and combines their answers.

Each call below prices one wine, so the cost is a fraction of a cent.

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(client)
agent.price(test[0].summary)

In [ ]:
from agents.neural_network_agent import NeuralNetworkAgent

agent = NeuralNetworkAgent()
agent.price(test[0].summary)

The EnsembleAgent takes any description, not only one in our format, because it runs the Preprocessor first.

In [ ]:
from agents.ensemble_agent import EnsembleAgent

agent = EnsembleAgent(client)
agent.price("A 2018 Barolo from Piedmont, Italy, made from Nebbiolo. 93 points. Rose petal, tar and red cherry, with firm tannins that need five more years.")

## Results so far

Put your scores in place of the zeros, then run the cell.

In [ ]:
results = [
    ("Constant", "gray", 21.71),
    ("XGBoost", "gray", 11.67),
    ("Deep Neural Network", "orange", 11.23),
    ("Qwen 3.8 27B", "slateblue", 14.42),
    ("Claude Sonnet 5", "slateblue", 9.06),
    ("Base Qwen 3 4B (4 bit)", "darkred", 23.15),
    ("Fine-tuned Qwen 3 4B", "red", 10.10),
    ("Qwen 3.8 27B RAG", "green", 9.96),
    ("GPT 5.1 RAG", "green", 8.87),
    ("Claude Sonnet 5 RAG", "green", 8.25),
    ("Ensemble", "purple", 7.96),
]

results = [result for result in results if result[2] > 0]
labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors, text=[f"${v:.2f}" for v in values], textposition="outside"))

fig.update_layout(
    title="Wine pricer - average error by model (lower is better)",
    yaxis=dict(range=[0, max(values) * 1.15], title="Error ($)"),
    xaxis=dict(tickangle=-45),
    width=1000,
    height=700
)

fig.show()

## Before you leave

Close the vector store, so that other notebooks and tomorrow's agents can open it.

In [ ]:
client.close()